# Import thư viện và Tải API Key

In [1]:
import pandas as pd
import requests
import os
import time
from tqdm import tqdm
from dotenv import load_dotenv

# Tải các biến môi trường từ file .env
load_dotenv(dotenv_path='../.env')

TMDB_API_KEY = os.getenv("TMDB_API_KEY")

if not TMDB_API_KEY:
    print("❌ Lỗi: Không tìm thấy TMDB_API_KEY. Hãy kiểm tra lại file .env!")
else:
    print("✅ Đã tải API Key thành công!")

✅ Đã tải API Key thành công!


# Đọc dữ liệu gốc từ MovieLens

In [2]:
# Đọc file dữ liệu (Đảm bảo bạn đã bỏ các file này vào thư mục data/raw/)
movies_df = pd.read_csv('../data/raw/ml-latest-small/movies.csv')
links_df = pd.read_csv('../data/raw/ml-latest-small/links.csv')

# Gộp 2 bảng lại với nhau dựa trên cột movieId
df = pd.merge(movies_df, links_df, on='movieId')

# Loại bỏ các dòng không có tmdbId (bị thiếu dữ liệu)
df = df.dropna(subset=['tmdbId']).copy()
df['tmdbId'] = df['tmdbId'].astype(int)

print(f"Tổng số phim cần lấy dữ liệu: {len(df)}")
df.head()

Tổng số phim cần lấy dữ liệu: 9734


,movieId,title,genres,imdbId,tmdbId
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862
1,2,Jumanji (1995),Adventure|Children|Fantasy,113497,8844
2,3,Grumpier Old Men (1995),Comedy|Romance,113228,15602
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,114885,31357
4,5,Father of the Bride Part II (1995),Comedy,113041,11862


# Định nghĩa hàm gọi API

In [3]:
def fetch_tmdb_metadata(tmdb_id):
    """Gọi API TMDB để lấy Overview, Thể loại, Từ khóa, Diễn viên và URL Poster"""
    url = f"https://api.themoviedb.org/3/movie/{tmdb_id}?api_key={TMDB_API_KEY}&append_to_response=keywords,credits"
    
    try:
        response = requests.get(url, timeout=10)
        
        # Nếu gọi thành công (Status code 200)
        if response.status_code == 200:
            data = response.json()
            
            # Trích xuất dữ liệu cần thiết
            overview = data.get('overview', '')
            genres = [g['name'] for g in data.get('genres', [])]
            keywords = [k['name'] for k in data.get('keywords', {}).get('keywords', [])]
            
            # Chỉ lấy 5 diễn viên đầu tiên để tránh loãng dữ liệu
            cast = [c['name'] for c in data.get('credits', {}).get('cast', [])[:5]] 
            
            poster_path = data.get('poster_path', '')
            
            return {
                'overview': overview,
                'tmdb_genres': genres,
                'keywords': keywords,
                'cast': cast,
                'poster_path': poster_path
            }
        else:
            return None # Trả về None nếu phim không tồn tại trên TMDB
            
    except Exception as e:
        # Bắt lỗi mạng hoặc timeout
        print(f"Lỗi khi lấy ID {tmdb_id}: {e}")
        return None

# Crawling

In [6]:
import concurrent.futures
print("Bắt đầu thu thập dữ liệu từ TMDB...")

metadata_list = []
# Chuyển dataframe thành danh sách các dictionary để dễ xử lý trong luồng
movies_to_fetch = df.to_dict('records')

# Hàm bọc (wrapper) để xử lý từng hàng
def process_movie(row):
    tmdb_id = row['tmdbId']
    metadata = fetch_tmdb_metadata(tmdb_id)
    if metadata:
        metadata['movieId'] = row['movieId']
        return metadata
    return None

# Sử dụng ThreadPoolExecutor để gọi API song song
# Lưu ý: TMDB giới hạn khoảng 40-50 requests/giây. 
# Tránh đặt max_workers quá cao để không bị đánh dấu là DDos tránh block IP.
with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
    # Dùng tqdm để tạo thanh tiến trình
    results = list(tqdm(executor.map(process_movie, movies_to_fetch), total=len(movies_to_fetch)))

# Lọc bỏ các kết quả None (phim không tồn tại trên TMDB)
metadata_list = [res for res in results if res is not None]
# Chuyển kết quả thành DataFrame
tmdb_metadata_df = pd.DataFrame(metadata_list)
print(f"\n✅ Đã lấy thành công dữ liệu của {len(tmdb_metadata_df)} bộ phim.")

Bắt đầu thu thập dữ liệu từ TMDB...


100%|██████████| 9734/9734 [27:16<00:00,  5.95it/s]


✅ Đã lấy thành công dữ liệu của 9621 bộ phim.


# Gộp dữ liệu và Lưu

In [7]:
# Gộp dữ liệu mới lấy được vào dataframe gốc
final_df = pd.merge(df, tmdb_metadata_df, on='movieId', how='inner')

# Lưu lại vào thư mục processed (đã được cấu hình gitignore bỏ qua)
output_path = '../data/processed/movies_enriched.csv'
final_df.to_csv(output_path, index=False, encoding='utf-8')

print(f"🎉 Đã lưu toàn bộ dữ liệu hoàn chỉnh tại: {output_path}")
final_df.head(3)

🎉 Đã lưu toàn bộ dữ liệu hoàn chỉnh tại: ../data/processed/movies_enriched.csv


,movieId,title,genres,imdbId,tmdbId,overview,tmdb_genres,keywords,cast,poster_path
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862,"Led by Woody, Andy's toys live happily in his ...","[Family, Comedy, Animation, Adventure]","[rescue, friendship, mission, jealousy, villai...","[Tom Hanks, Tim Allen, Don Rickles, Jim Varney...",/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg
1,2,Jumanji (1995),Adventure|Children|Fantasy,113497,8844,When siblings Judy and Peter discover an encha...,"[Adventure, Fantasy, Family]","[giant insect, board game, disappearance, jung...","[Robin Williams, Kirsten Dunst, Bradley Pierce...",/vgpXmVaVyUL7GGiDeiK1mKEKzcX.jpg
2,3,Grumpier Old Men (1995),Comedy|Romance,113228,15602,A family wedding reignites the ancient feud be...,"[Romance, Comedy]","[fishing, sequel, old man, best friend, weddin...","[Walter Matthau, Jack Lemmon, Ann-Margret, Sop...",/1FSXpj5e8l4KH6nVFO5SPUeraOt.jpg
